In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
import statsmodels.api as sm
import pandas as pd

def compute_eci(orders_df, complements_df, output_csv=None):

    if output_csv:
        with open(output_csv, 'w') as f:
            f.write("product_id,complement_id,coef,p_value,pseudo_r2,ame,eci_units,complement_units,complement_increase\n")

    # Map orders and products to indices
    orders_df['order_idx'] = orders_df['order_id'].astype('category').cat.codes
    orders_df['product_idx'] = orders_df['product_id'].astype('category').cat.codes
    product_idx_map = dict(enumerate(orders_df['product_id'].astype('category').cat.categories))

    n_orders = orders_df['order_idx'].max() + 1
    n_products = orders_df['product_idx'].max() + 1

    # Sparse basket matrix
    data = np.ones(len(orders_df), dtype=np.uint8)
    basket_matrix = csr_matrix((data, (orders_df['order_idx'], orders_df['product_idx'])),
                               shape=(n_orders, n_products))

    # Order-level features
    order_features = orders_df[['order_idx', 'order_size_scaled',
                                'orders_per_user_scaled', 'prod_per_user_scaled']].drop_duplicates('order_idx').set_index('order_idx')

    results = []

    for product_id, group in complements_df.groupby('product_id'):

        idx_target = {v: k for k, v in product_idx_map.items()}.get(product_id)
        if idx_target is None:
            continue

        orders_target_idx = set(basket_matrix[:, idx_target].nonzero()[0])
        if not orders_target_idx:
            continue

        for _, row in group.iterrows():
            comp_id = row['complement_id']
            idx_comp = {v: k for k, v in product_idx_map.items()}.get(comp_id)
            if idx_comp is None:
                continue

            y = basket_matrix[:, idx_comp].toarray().ravel()
            target_col = basket_matrix[:, idx_target].toarray().ravel()

            # Feature matrix
            X = order_features.copy()
            X['target_present'] = target_col
            X['lift'] = row.get('lift', 0)
            X = sm.add_constant(X)

            try:
                model = sm.Logit(y, X).fit(disp=0)
            except Exception:
                continue

            # Compute AME (correct sign)
            X_present = X.copy(); X_present['target_present'] = 1
            X_absent = X.copy(); X_absent['target_present'] = 0
            pred_present = model.predict(X_present)
            pred_absent = model.predict(X_absent)
            ame = (pred_present - pred_absent).mean()

            # ECI units (effect on orders containing target)
            support_A = target_col.sum()
            eci_units = ame * support_A

            # Complement increase in percentage points (absolute probability increase)
            complement_increase = ame * 100  # interpretable as % points increase

            # Safe pseudo-R2
            llnull = getattr(model, 'llnull', None)
            pseudo_r2 = np.nan if (llnull is None or np.isclose(llnull, 0.0)) else 1 - (model.llf / llnull)

            result = {
                'product_id': product_id,
                'complement_id': comp_id,
                'coef': model.params['target_present'],
                'p_value': model.pvalues['target_present'],
                'pseudo_r2': pseudo_r2,
                'ame': ame,
                'eci_units': eci_units,
                'complement_units': len(y),
                'complement_increase': complement_increase
            }

            results.append(result)

            if output_csv:
                pd.DataFrame([result]).to_csv(output_csv, mode='a', index=False, header=False)

    return pd.DataFrame(results)

complements_df = pd.read_csv("../data/results/complements-lift.csv")
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')
order_features_df = pd.read_csv('../data/cleaned/order-features.csv')  
orders_df = orders_full_df.merge(order_features_df, on='order_id', how='left')

#chosen_ids = comps_df['product_id'].drop_duplicates().sample(5, random_state=42)
#complements_df = comps_df[comps_df['product_id'].isin(chosen_ids)]

complements_df['lift'] = pd.to_numeric(complements_df['lift'], errors='coerce')
#top_complements_df = complements_df.nlargest(100, 'lift') 

complements_df.rename(columns={'product_i': 'product_id', 'product_j': 'complement_id'}, inplace=True)


compute_eci(orders_df, 
    complements_df, 
    output_csv="../data/results/complements-impact.csv"
)